<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_1_train_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1. Entrenamiento de modelo Random Forest


## 0. Clonado de Repositorio, instalación de librería e importación.

### Clonado de Repositorio

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


### Acceso de Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Instalación de librerías

In [3]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

/bin/bash: line 1: {sys.executable}: command not found
Librerías instaladas: ta


### Importación de librerías

In [4]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
#import ta
#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

### Carga de mnq_model

In [5]:
def load_data_from_drive():

    # Definir la URL del archivo Parquet en Drive
    df_path_mnq = f'{drive_path}/mnq_data/mnq_model.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)

    return df_model

In [6]:
mnq_model = load_data_from_drive()

## 1. Carga de ventanas X_* escalad, y_* y el  escalador

In [7]:
import numpy as np
import joblib

def load_datasets_and_scaler(drive_path, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """
    folder = "ventanas_x_y_scaled" if scaled else "ventanas_x_y"
    scaled_name = "_scaled" if scaled else ""

    ruta_train  = f"{drive_path}/{folder}/mnq_Xy_train{scaled_name}.npz"
    ruta_valid  = f"{drive_path}/{folder}/mnq_Xy_valid{scaled_name}.npz"
    ruta_test   = f"{drive_path}/{folder}/mnq_Xy_test{scaled_name}.npz"


    ruta_scaler = f"{drive_path}/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(ruta_train)
    data_valid = np.load(ruta_valid)
    data_test  = np.load(ruta_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(ruta_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler

In [8]:
X_train_s, y_train, X_valid_s, y_valid, X_test_s, y_test, scaler = load_datasets_and_scaler(drive_path)

## 2. Selección y filtrado de features

In [9]:
 #Listado de features para el modelo
 features_random_forest = ['factor30', 'rsi_14', 'price_ema30', 'stoch_k_20', 'bb_percent_30_20', 'reversal_momentum_factor', 'rsi_7', 'reversal_media_factor', 'rsi_3', 'bb_percent_20_15']

In [10]:
    #Listado de features sin 'date'
    features =  mnq_model.columns.tolist()
    features.remove('date') #Remover fecha
    features.remove('target_return_30')  #Remover el target

Función para filtrar features de los X_*, siempre con los datos escalados

In [11]:
def filter_features_to_model(
    features_select,          #Listado de features a mantener
    X_train = X_train_s,    #Siempre usaremos los subsets escalados
    X_valid = X_valid_s,
    X_test = X_test_s,
    features = features,    #El listado de features siempre es el mismo.
    window_size = 60        #Siempre el mismo
        ):

    # Índices fijos de OHLCV
    idx_list = [0, 1, 2, 3, 4]

    for name in features_select:
        if name in features:
            idx_list.append(features.index(name))
        else:
            print(f"⚠️ Feature '{name}' no encontrado en features_list.")

    # Ordenar y eliminar duplicados
    idx_features_selected = sorted(set(idx_list))

    n_features = len(features)

    # Verificación rápida
    n_total_cols = window_size * n_features
    assert X_train.shape[1] == n_total_cols, "X_train no coincide con window_size * n_features"

    # Calcular columnas a mantener
    cols_to_keep = []
    for idx in idx_features_selected:
        start = idx * window_size
        end = (idx + 1) * window_size
        cols_to_keep.extend(range(start, end))

    # Filtrar arrays
    X_train_f = X_train[:, cols_to_keep]
    X_valid_f = X_valid[:, cols_to_keep]
    X_test_f  = X_test[:, cols_to_keep]

    # Features filtrados
    features_f = [features[i] for i in idx_features_selected]

    print(f"Features seleccionados ({len(features_f)}): {features_f}")
    print("X_train_f shape:", X_train_f.shape)
    print("X_valid_f shape:", X_valid_f.shape)
    print("X_test_f shape :", X_test_f.shape)

    return X_train_f, X_valid_f, X_test_f, features_f

In [12]:
X_train_model, X_valid_model, X_test_model, features_model = filter_features_to_model(
    features_select = features_random_forest,
)

Features seleccionados (15): ['open', 'high', 'low', 'close', 'volume', 'rsi_3', 'rsi_7', 'rsi_14', 'stoch_k_20', 'bb_percent_20_15', 'bb_percent_30_20', 'price_ema30', 'reversal_momentum_factor', 'reversal_media_factor', 'factor30']
X_train_f shape: (276017, 900)
X_valid_f shape: (59297, 900)
X_test_f shape : (59297, 900)


In [13]:
features_model

['open',
 'high',
 'low',
 'close',
 'volume',
 'rsi_3',
 'rsi_7',
 'rsi_14',
 'stoch_k_20',
 'bb_percent_20_15',
 'bb_percent_30_20',
 'price_ema30',
 'reversal_momentum_factor',
 'reversal_media_factor',
 'factor30']

## 4. Entrenamiento de modelo

In [14]:
# --- IMPORTS & VERSION CHECK ---
import os, json, joblib, numpy as np, pandas as pd
import sklearn, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")

print("Python:", __import__("platform").python_version())
print("scikit-learn:", sklearn.__version__)  # -> debería ser 1.4.2

RANDOM_SEED = 42
rng = np.random.RandomState(RANDOM_SEED)

Python: 3.12.11
scikit-learn: 1.6.1


### 4.2. Verificar tipo de variable y dimensiones de X_train_model, X_valid_model y X_test_model

In [15]:
def ensure_numpy(X, fallback_features=None):
    """
    Si X ya es ndarray, lo devuelve tal cual y devuelve None como columnas.
    Si X es DataFrame, devuelve .values y las columnas.
    """
    import numpy as np
    import pandas as pd

    if isinstance(X, np.ndarray):
        return X, None
    elif isinstance(X, pd.DataFrame):
        return X.values, list(X.columns)
    else:
        raise TypeError(f"Tipo no soportado: {type(X)}")

# Conversión segura
X_train_s, cols_train = ensure_numpy(X_train_model)
X_valid_s, cols_valid = ensure_numpy(X_valid_model)
X_test_s,  cols_test  = ensure_numpy(X_test_model)

# Chequeo de consistencia solo si hay nombres de columnas
if cols_train and cols_valid and cols_test:
    assert cols_train == cols_valid == cols_test, "El orden/columns difiere entre splits."
    feature_names = cols_train
    print('X_* son pd.DataFrame (se verificaron columnas)')
else:
    feature_names = features_model  # lista fija que ya definiste
    print(f'X_* son numpy.ndarray, se usa la lista fija de features: \t{features_model}')
    #print(features_model)

# Convertir y_*
import numpy as np
y_tr = np.asarray(y_train)
y_va = np.asarray(y_valid)
y_te = np.asarray(y_test)

print("\nShapes:", X_train_s.shape, X_valid_s.shape, X_test_s.shape)

X_* son numpy.ndarray, se usa la lista fija de features: 	['open', 'high', 'low', 'close', 'volume', 'rsi_3', 'rsi_7', 'rsi_14', 'stoch_k_20', 'bb_percent_20_15', 'bb_percent_30_20', 'price_ema30', 'reversal_momentum_factor', 'reversal_media_factor', 'factor30']

Shapes: (276017, 900) (59297, 900) (59297, 900)


### 4.3. Definición de modelo base y el espacio de búsqueda

In [16]:
rf = RandomForestRegressor(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

#Definimos espacio de búsqueda donde el algoritmo probará combinaciones de hiperparámetros:

param_distributions = {
    "n_estimators": randint(300, 1200),                 #Número de árboles → más árboles suelen mejorar el resultado, pero tardan más.
    "max_depth": [None] + list(range(4, 40)),       #Qué tan profundo puede crecer cada árbol → controla el sobreajuste.
    "max_features": ['sqrt', 'log2', 0.5, 0.7, 1.0],  #Cuántas columnas usar en cada división → ayuda a diversificar árboles.
    "min_samples_split": randint(2, 20),                #Mínimo de muestras para dividir o quedar en una hoja → también controla sobreajuste.
    "min_samples_leaf": randint(1, 20),
    "bootstrap": [True, False],                                    #Si entreno cada árbol con muestras aleatorias con reemplazo (True) o con todo el dataset (False).
    "ccp_alpha": uniform(0.0, 0.02),                        #poda → recorta árboles muy complejos.
}


In [ ]:
from sklearn.model_selection import GroupKFold
import numpy as np

DAY_LEN = 361  # minutos por día en tu dataset

# groups_train con la MISMA longitud que X_train_model:
groups_train = np.arange(len(X_train_model)) // DAY_LEN

print("Len X_train:", len(X_train_model))
print("Len y_train:", len(y_train))
print("Len groups_train:", len(groups_train))
print("Últimos 10 groups:", groups_train[-10:])

cv = GroupKFold(n_splits=3)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=60,
    scoring="neg_root_mean_squared_error",
    refit=True,
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train_model, y_train, groups=groups_train)

Len X_train: 276017
Len y_train: 276017
Len groups_train: 276017
Últimos 10 groups: [764 764 764 764 764 764 764 764 764 764]
Fitting 3 folds for each of 60 candidates, totalling 180 fits


# LISTO HASTA ACÁ!

In [ ]:
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=60,                          # ajustá si querés acelerar
    scoring="neg_root_mean_squared_error",
    refit=True,
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)

## CODIGO

In [ ]:
# ================================
# 2) Convertimos a numpy (sin re-escalar)
# ================================
X_train_s, cols_train = to_numpy(X_train_model)
X_valid_s, cols_valid = to_numpy(X_valid_model)
X_test_s,  cols_test  = to_numpy(X_test_model)

# chequeo simple de compatibilidad columnas
if cols_train and cols_valid and cols_test:
    assert cols_train == cols_valid == cols_test, "El orden/columns difiere entre splits."
feature_names = cols_train or SELECTED_FEATURES

y_tr = np.asarray(y_train)
y_va = np.asarray(y_valid)
y_te = np.asarray(y_test)

print("Shapes:", X_train_s.shape, X_valid_s.shape, X_test_s.shape)

# ================================
# 3) Definimos RF + espacio de búsqueda
# ================================
rf = RandomForestRegressor(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

param_distributions = {
    "n_estimators": randint(300, 1200),
    "max_depth": [None] + list(range(4, 40)),
    "max_features": ['sqrt', 'log2', 0.5, 0.7, 1.0],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 20),
    "bootstrap": [True, False],
    "ccp_alpha": uniform(0.0, 0.02),
}

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)  # cambia a GroupKFold si tenés day_id

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=60,                          # ajustá si querés acelerar
    scoring="neg_root_mean_squared_error",
    refit=True,
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)

# ================================
# 4) Entrenar
# ================================
search.fit(X_train_s, y_tr)
best_rf = search.best_estimator_
print("Mejores hiperparámetros:", search.best_params_)

# ================================
# 5) Métricas
# ================================
metrics = {
    "train": evaluate(best_rf, X_train_s, y_tr),
    "valid": evaluate(best_rf, X_valid_s, y_va),
    "test":  evaluate(best_rf,  X_test_s,  y_te),
}
print("Métricas:", json.dumps(metrics, indent=2))

# ================================
# 6) Importancias de variables
# ================================
importancias = pd.Series(best_rf.feature_importances_, index=feature_names)\
                .sort_values(ascending=False)
print(importancias.head(15))

# ================================
# 7) Guardado de modelo + metadatos (sin scaler)
# ================================
MODEL_DIR = "/content/neural_profit/models"
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, "rf_random_target_return_30_v1.joblib")
META_PATH  = os.path.join(MODEL_DIR, "rf_random_target_return_30_v1_meta.json")

joblib.dump(best_rf, MODEL_PATH)

meta = {
    "name": "rf_random_target_return_30_v1",
    "framework": "scikit-learn",
    "sklearn_version": sklearn.__version__,
    "random_seed": RANDOM_SEED,
    "features": feature_names,          # el orden que usaste
    "best_params": search.best_params_,
    "metrics": metrics,
    "model_path": MODEL_PATH,
    "scaled_inputs": True               # importante dejarlo explícito
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Guardados:")
print(MODEL_PATH)
print(META_PATH)